# 01 — Health

**Is the cluster there and wired?** Four checks, under a minute, no load and
no interpretation. The only output that matters is the last line: safe to
proceed, or not.

| | |
|---|---|
| **topology** | is every component deployed, at the replica count we expect |
| **health** | is each one ready, and has it been restarting |
| **wiring** | does a request complete, and does it go *through* the gateway |
| **observability** | is every metric family actually being scraped |

What this is **not**: any claim about performance. No percentiles, no SLOs,
no load. `02-metrics` reads what the numbers mean; `03-capacity` pushes them.

**Needs** `make ssh`. Independent of every other notebook.


In [ ]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.insert(0, '..')

from utils import Cluster, Report
from prom import Prom, EXPECTED_DEPLOYMENTS, METRIC_FAMILIES, deployments

cluster = Cluster()
prom    = Prom()
report  = Report('Dodona 01 — health', 'is it there and wired?')

report.objective(
    'Is every component deployed, healthy, in the request path, and being '
    'scraped?',
    falsified_by='a missing deployment, an unready pod, a request that does not '
                 'reach the gateway, or a metric family with no series')


## 1. Topology — is everything deployed?

Read from **kube-state-metrics**, so this reflects what the cluster reports
rather than what a manifest claims. Those differ whenever an apply silently
did nothing.


In [ ]:
report.heading('1. Topology')
found = deployments(prom)
rows, wrong = [], []
for name, what in EXPECTED_DEPLOYMENTS.items():
    desired, available, restarts = found[name]
    ok = available >= 1 and available == desired
    rows.append([name, f'{available:.0f}/{desired:.0f}', f'{restarts:.0f}', what])
    if not ok:
        wrong.append(name)
report.table(['deployment', 'ready', 'restarts', 'what it does'], rows,
             note='restarts above 0 means it has crashed at least once.')
report.verdict(not wrong, f'all {len(EXPECTED_DEPLOYMENTS)} deployments are at their replica count')

flapping = [n for n, (_, _, r) in found.items() if r > 0]
report.verdict(not flapping, 'no deployment has restarted')
report.conclusion(
    'Every component is deployed and none has crashed.' if not wrong and not flapping
    else (f'Not at count: {", ".join(wrong)}. ' if wrong else '')
       + (f'Has restarted: {", ".join(flapping)} — check `make logs` before '
          'trusting any measurement, a pod that restarted mid-run has reset counters.'
          if flapping else ''))


## 2. Health — does everything answer?

`Ready` in Kubernetes means a probe passed. This asks the endpoints directly,
which is what a client experiences.


In [ ]:
report.heading('2. Health')
up = cluster.endpoints()
up['prometheus'] = prom.up()
for name, ok in up.items():
    report.verdict(ok, name)
down = [n for n, ok in up.items() if not ok]
report.conclusion(
    'Every endpoint answers.' if not down else
    f'Unreachable: {", ".join(down)}. `make ssh` for the tunnel, '
    '`make observability` for Prometheus and Grafana.')


## 3. Wiring — is the control plane in the path?

The check that decides whether anything we built is real. `orch_requests_total`
only moves if the request passed our gateway; a completion that reached a pod
directly leaves it untouched, and from outside the two look identical.


In [ ]:
report.heading('3. Wiring')
from utils import sum_by_name
before = sum_by_name(cluster.gateway_metrics(), 'orch_requests_total')
answer = cluster.chat('Reply with the single word: ok', max_tokens=8)
after  = sum_by_name(cluster.gateway_metrics(), 'orch_requests_total')

report.verdict(after > before, f'orch_requests_total {before:.0f} -> {after:.0f}')
report.verdict('_error' not in answer, f"a request completed in {answer['_latency_s']}s")

if '_error' in answer:
    report.text(f"{answer['_error']}: {str(answer.get('_body',''))[:180]}")
    report.conclusion('The gateway answered but refused. That is a gateway '
                      'decision, not a GPU limit — the reason is above.')
else:
    said = ((answer.get('choices') or [{}])[0].get('message') or {}).get('content')
    hop = answer.get('kv_hop')
    report.table(['', ''], [['answered', repr(said)], ['via', answer.get('via')],
                            ['kv_hop recorded', bool(hop)]])
    report.conclusion(
        f'A request went guardrails to admission to placement to the engines and '
        f'came back {said!r}. via={answer.get("via")} means our cluster served it.')


## 4. Observability — is it all being scraped?

An empty Grafana panel and a missing scrape target look identical. Zero
series for a family means **nothing exports it** — which is why a panel can
be empty for a reason that has nothing to do with the cluster.


In [ ]:
report.heading('4. Observability')
rows, missing = [], []
for prefix, who in METRIC_FAMILIES.items():
    n = prom.series_exist(prefix)
    rows.append([prefix, n, who])
    if n == 0:
        missing.append(prefix)
report.table(['prefix', 'series', 'emitted by'], rows,
             note='0 means nothing scrapes it, not that the value is zero.')
report.verdict(not missing, f'all {len(METRIC_FAMILIES)} metric families are scraped')
report.conclusion(
    'Every family is present, so an empty panel from here on means a real zero.'
    if not missing else
    f'Missing: {", ".join(missing)}. Anything built on those reads empty for a '
    'reason unrelated to how the cluster is behaving.')


## Verdict


In [ ]:
path  = report.save('01-health')
fails = report.failures()
print('wrote', path)
print()
if fails:
    print('NOT SAFE TO PROCEED — fix these first:')
    for f in fails:
        print('  -', f)
else:
    print('Cluster is there and wired. Next: 02-metrics')
